# Week 2 Day 2 Exercise: SDR email pipeline over SMTP

Starts from the code in `2_lab2.ipynb` (SMTP email, the three sales writers, the picker, the send tool).

Added on top:
- **Refiner agent**: polishes the chosen email into a master template.
- **Mail merge agent**: personalises that template for each prospect in `prospects.csv`.
- **Automation**: prospects come from a CSV, every send is logged to `send_log.csv`.
- **Three orchestration styles** over the same pipeline:
  1. Orchestrate by **code** (plain Python decides the steps).
  2. Orchestrate by **LLM with tools** (a manager agent calls the other agents as tools).
  3. Orchestrate by **LLM with handoffs** (a manager hands the send step to a sender agent).

Safety: `SEND_TO_SELF = True` sends every email to your own `EMAIL_ADDRESS`. Set it to `False` only when you are happy for real prospects to receive mail.

Run from the `community_contributions` folder so `prospects.csv` and `send_log.csv` land next to this notebook.


## Part 1: Setup


In [ ]:
import os
import csv
import json
import asyncio
import smtplib
from email.message import EmailMessage

from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool, ModelSettings

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"
SEND_TO_SELF = True

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

print("Email set up:", all([EMAIL_ADDRESS, EMAIL_SMTP_SERVER, EMAIL_APP_PASSWORD]))


## Part 2: Shared pieces (send, prospects, output shapes)


In [ ]:
class Email(BaseModel):
    subject: str
    text_body: str
    html_body: str


class Pick(BaseModel):
    choice: int  # 1-based number of the best draft


def send_email(to_addr, subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS if SEND_TO_SELF else to_addr
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

    new_file = not os.path.exists("send_log.csv")
    with open("send_log.csv", "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["to", "subject"])
        writer.writerow([msg["To"], subject])
    return f"sent to {msg['To']}"


PROSPECTS_FILE = "prospects.csv"
if not os.path.exists(PROSPECTS_FILE):
    # Starter list. Edit prospects.csv to use your own leads; the pipeline reads it on every run.
    with open(PROSPECTS_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["first_name", "company", "role", "email"])
        writer.writeheader()
        writer.writerows([
            {"first_name": "Ada", "company": "Lovelace Labs", "role": "Head of Compliance", "email": "ada@example.com"},
            {"first_name": "Grace", "company": "Hopper Systems", "role": "CTO", "email": "grace@example.com"},
            {"first_name": "Linus", "company": "Torvald Cloud", "role": "VP Engineering", "email": "linus@example.com"},
        ])


def load_prospects(path=PROSPECTS_FILE):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


prospects = load_prospects()
print(f"{len(prospects)} prospects loaded")


## Part 3: The agents

- Three **writers** with different styles (from the lab).
- A **picker** that chooses the best draft.
- A **refiner** that polishes the chosen draft and adds `{first_name}` and `{company}` placeholders.
- A **mail merge** agent that fills those placeholders for one prospect and rewrites the opening line for their role.


In [ ]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write cold sales emails.
"""

writer_instructions = [
    intro + "Your email style is professional, serious, with gravitas and credibility.",
    intro + "Your email style is witty, engaging, and humorous.",
    intro + "Your email style is concise, to the point, in the style of a busy senior executive.",
]

sales_writers = [
    Agent(name=f"Writer {i + 1}", instructions=ins, model=MODEL_NAME, output_type=Email)
    for i, ins in enumerate(writer_instructions)
]

picker = Agent(
    name="Draft Picker",
    model=MODEL_NAME,
    output_type=Pick,
    instructions="You pick the cold email you, as a busy prospect, are most likely to reply to. Return only its number.",
)

refiner = Agent(
    name="Email Refiner",
    model=MODEL_NAME,
    output_type=Email,
    instructions="""
You improve a cold sales email. Tighten the copy, cut clichés, and end with one clear, low-pressure call to action.
Keep the facts the same and keep the body under 150 words.
Use the placeholders {first_name} in the greeting and {company} once in the body, so a mail merge can fill them.
Return the full email: subject, text_body and html_body.
""",
)

merge_agent = Agent(
    name="Mail Merge Agent",
    model=MODEL_NAME,
    output_type=Email,
    instructions="""
You do a mail merge. You get a master email (a template) and one prospect record as JSON.
Fill every placeholder ({first_name}, {company}, {role}) with the prospect's values.
Rewrite only the first sentence of the body so it speaks to the prospect's role. Keep everything else.
Do not invent facts about the prospect or their company.
Return subject, text_body and html_body.
""",
)


## Part 4: Shared tool

The send step is a plain function tool. Every orchestration style below uses it.


In [ ]:
@function_tool
def send_email_tool(to_addr: str, subject: str, text_body: str, html_body: str) -> str:
    """Send one email to a prospect. to_addr is the prospect's email address."""
    return send_email(to_addr, subject, text_body, html_body)


## Part 5: Orchestration by code

Python decides every step. Predictable and easy to debug. Writers run in parallel, then the picker, refiner, and one mail merge per prospect.


In [ ]:
async def orchestrate_by_code(brief, prospect_list):
    with trace("Exercise: orchestrate by code"):
        results = await asyncio.gather(*[Runner.run(w, brief) for w in sales_writers])
        drafts = [r.final_output for r in results]

        numbered = "\n\n".join(
            f"Draft {i + 1}:\nSubject: {d.subject}\n{d.text_body}" for i, d in enumerate(drafts)
        )
        pick = (await Runner.run(picker, numbered)).final_output
        best = drafts[min(max(pick.choice, 1), len(drafts)) - 1]

        refined = (await Runner.run(refiner, best.model_dump_json())).final_output

        for p in prospect_list:
            merged = (await Runner.run(
                merge_agent,
                f"Template:\n{refined.model_dump_json()}\n\nProspect:\n{json.dumps(p)}",
            )).final_output
            print(send_email(p["email"], merged.subject, merged.text_body, merged.html_body))

    return refined


In [ ]:
brief = "Write a cold sales email to a senior compliance leader about getting ready for a SOC2 audit faster."

refined_email = await orchestrate_by_code(brief, prospects)
print("Master template subject:", refined_email.subject)


## Part 6: Orchestration by LLM with tools

One manager agent decides what to do. The writers, refiner, and mail merge are exposed as tools with `as_tool`. The manager keeps control and calls each tool in turn.


In [ ]:
writer_tools = [
    w.as_tool(
        tool_name=f"write_email_{i + 1}",
        tool_description="Write one cold sales email. Pass the brief as input.",
    )
    for i, w in enumerate(sales_writers)
]

refine_tool = refiner.as_tool(
    tool_name="refine_email",
    tool_description="Polish a chosen cold email. Pass the full email as input.",
)

merge_tool = merge_agent.as_tool(
    tool_name="mail_merge",
    tool_description="Personalise the master template for one prospect. Pass the template and the prospect JSON.",
)

tools_manager = Agent(
    name="Sales Manager (tools)",
    model=MODEL_NAME,
    tools=[*writer_tools, refine_tool, merge_tool, send_email_tool],
    instructions="""
You run an outbound email campaign for ComplAI.
1. Call write_email_1, write_email_2 and write_email_3 with the brief. Wait for all three.
2. Pick the single best draft yourself.
3. Call refine_email with that draft.
4. For each prospect you were given: call mail_merge with the refined template and the prospect's JSON, then call send_email_tool with the prospect's email address and the merged email.
Stop after every prospect has been sent. Reply with a one-line summary.
""",
)


In [ ]:
with trace("Exercise: orchestrate by LLM with tools"):
    result = await Runner.run(
        tools_manager,
        input=f"Brief: {brief}\n\nProspects:\n{json.dumps(prospects, indent=2)}",
    )
print(result.final_output)


## Part 7: Orchestration by LLM with handoffs

The manager writes drafts with tools, then **hands off** to a sender agent that owns refining and sending. Control passes across rather than returning.

The lab noted handoffs can be unreliable. `tool_choice="required"` on the sender forces it to act instead of replying in text.


In [ ]:
sender = Agent(
    name="Sender",
    model=MODEL_NAME,
    tools=[refine_tool, merge_tool, send_email_tool],
    model_settings=ModelSettings(tool_choice="required"),
    instructions="""
You own sending. You get the chosen draft and the prospect list.
1. Call refine_email with the draft.
2. For each prospect: call mail_merge with the refined template and the prospect JSON, then call send_email_tool with the prospect's email address.
Only email prospects from the list you were given.
""",
)

handoff_manager = Agent(
    name="Sales Manager (handoffs)",
    model=MODEL_NAME,
    tools=writer_tools,
    handoffs=[sender],
    instructions="""
You run an outbound email campaign for ComplAI.
1. Call write_email_1, write_email_2 and write_email_3 with the brief. Wait for all three.
2. Pick the single best draft yourself.
3. Hand off to Sender. Pass along the chosen draft and the full prospect list.
""",
)


In [ ]:
with trace("Exercise: orchestrate by LLM with handoffs"):
    result = await Runner.run(
        handoff_manager,
        input=f"Brief: {brief}\n\nProspects:\n{json.dumps(prospects, indent=2)}",
    )
print(result.final_output)


## Part 8: Check the results

Check the traces at https://platform.openai.com/traces, your inbox (and spam folder), and `send_log.csv`.

Each run above sends one email per prospect, so the three runs send up to nine emails to you. Remove prospects from `prospects.csv` to test with fewer.
